# Morris Clustering

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import os

# ===== 사용자 경로 설정 =====
base_path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA"
csv_path = os.path.join(base_path, "morrisresultcsv.csv")

# ===== 데이터 불러오기 =====
df = pd.read_csv(csv_path)

# ParNum이 있다면 유지
if 'ParNum' in df.columns:
    par_col = 'ParNum'
else:
    # 인덱스로 대체
    df['ParNum'] = range(1, len(df) + 1)
    par_col = 'ParNum'

# ===== 결과 폴더 생성 (avg, max, min) =====
for sub in ["avg", "max", "min"]:
    os.makedirs(os.path.join(base_path, sub), exist_ok=True)

summary_rows = []  # 요약 저장용 리스트

# ===== 연도 및 타입 반복 =====
years = range(2011, 2021)
types = ["avg", "max", "min"]

for t in types:
    print(f"\n📊 Processing {t.upper()} ...")

    for year in years:
        x_col = f"X_{year}{t}"
        y_col = f"Y_{year}{t}"

        if not (x_col in df.columns and y_col in df.columns):
            print(f"⚠️ {x_col} or {y_col} not found — skip.")
            continue

        # 데이터 준비
        data = df[[par_col, x_col, y_col]].copy()
        data = data[(data[x_col] != 0) & (data[y_col] != 0)].dropna().reset_index(drop=True)

        if len(data) < 3:
            print(f"⚠️ Not enough data for {year}-{t}, skip.")
            continue

        # ===== Median 기준 =====
        x_med = data[x_col].median()
        y_med = data[y_col].median()

        def classify_median(row):
            if row[x_col] >= x_med and row[y_col] >= y_med:
                return "High importance & High interaction"
            elif row[x_col] >= x_med and row[y_col] < y_med:
                return "High importance & Low interaction"
            elif row[x_col] < x_med and row[y_col] >= y_med:
                return "Low importance & High interaction"
            else:
                return "Low importance & Low interaction"

        data['Quadrant_Group'] = data.apply(classify_median, axis=1)

        # ===== Percentile 기준 (70%) =====
        x_thr = data[x_col].quantile(0.7)
        y_thr = data[y_col].quantile(0.7)

        def classify_quantile(row):
            if row[x_col] >= x_thr and row[y_col] >= y_thr:
                return "High importance & High interaction"
            elif row[x_col] >= x_thr and row[y_col] < y_thr:
                return "High importance & Low interaction"
            elif row[x_col] < x_thr and row[y_col] >= y_thr:
                return "Low importance & High interaction"
            else:
                return "Low importance & Low interaction"

        data['Quantile70_Group'] = data.apply(classify_quantile, axis=1)

        # ===== K-means =====
        kmeans = KMeans(n_clusters=3, random_state=42)
        data['Cluster'] = kmeans.fit_predict(data[[x_col, y_col]])

        # ===== 색상 설정 =====
        colors = {
            "High importance & High interaction": "red",
            "High importance & Low interaction": "orange",
            "Low importance & High interaction": "blue",
            "Low importance & Low interaction": "gray"
        }

        # ===== 그래프 저장 함수 =====
        def plot_groups(x_thr, y_thr, label_col, title_suffix, save_name):
            plt.figure(figsize=(7,6))
            for grp, color in colors.items():
                subset = data[data[label_col] == grp]
                plt.scatter(subset[x_col], subset[y_col],
                            label=f"{grp} ({len(subset)})", color=color, s=40, alpha=0.8)
            plt.axvline(x=x_thr, color='black', linestyle='--', linewidth=1)
            plt.axhline(y=y_thr, color='black', linestyle='--', linewidth=1)
            plt.title(f"{year} ({t}) - {title_suffix}")
            plt.xlabel(f"{x_col} (μ*, Importance)")
            plt.ylabel(f"{y_col} (σ, Interaction)")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            save_path = os.path.join(base_path, t, save_name)
            plt.savefig(save_path, dpi=300)
            plt.close()

        # ===== 피규어 1: Median 기준 =====
        plot_groups(x_med, y_med, 'Quadrant_Group', 'Median-based Quadrant',
                    f"{year}_{t}_quadrant.png")

        # ===== 피규어 2: 70% 퍼센타일 기준 =====
        plot_groups(x_thr, y_thr, 'Quantile70_Group', '70th Percentile',
                    f"{year}_{t}_quantile70.png")

        # ===== 피규어 3: K-means =====
        plt.figure(figsize=(7,6))
        plt.scatter(data[x_col], data[y_col], c=data['Cluster'], cmap='viridis', s=40, alpha=0.8)
        plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1],
                    c='red', marker='x', s=120, label='Centers')
        plt.title(f"{year} ({t}) - K-means Clustering")
        plt.xlabel(f"{x_col} (μ*, Importance)")
        plt.ylabel(f"{y_col} (σ, Interaction)")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(base_path, t, f"{year}_{t}_kmeans.png"), dpi=300)
        plt.close()

        # ===== 그룹별 요약 저장 =====
        for colname in ['Quadrant_Group', 'Quantile70_Group', 'Cluster']:
            groups = data.groupby(colname)[par_col].apply(list)
            for group_name, par_list in groups.items():
                summary_rows.append({
                    "Year": year,
                    "Type": t,
                    "Method": colname,
                    "Group": group_name,
                    "Count": len(par_list),
                    "ParNums": ','.join(map(str, par_list))
                })

        print(f"✅ {year}-{t} 완료 (그래프 3개 저장됨)")

# ===== 전체 요약 저장 =====
summary_df = pd.DataFrame(summary_rows)
summary_csv_path = os.path.join(base_path, "morris_summary_groups.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"\n📁 그룹 요약 CSV 저장 완료: {summary_csv_path}")



📊 Processing AVG ...


c:\Users\sl177\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


✅ 2011-avg 완료 (그래프 3개 저장됨)


c:\Users\sl177\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


✅ 2012-avg 완료 (그래프 3개 저장됨)


...

✅ 2018-min 완료 (그래프 3개 저장됨)


c:\Users\sl177\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


✅ 2019-min 완료 (그래프 3개 저장됨)


c:\Users\sl177\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


✅ 2020-min 완료 (그래프 3개 저장됨)

📁 그룹 요약 CSV 저장 완료: C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\morris_summary_groups.csv


# 각 폴더별(AVG/MAX/MIN) 연도별 피규어 합쳐 PDF 저장

In [17]:
import os
from PIL import Image
import glob

# ===== 기본 경로 =====
base_path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA"

# ===== 타입별 폴더 및 방법별 키워드 =====
types = ["avg", "max", "min"]
methods = ["quadrant", "quantile70", "kmeans"]

for t in types:
    folder = os.path.join(base_path, t)
    if not os.path.exists(folder):
        print(f"⚠️ {folder} 폴더 없음, 건너뜀.")
        continue

    for method in methods:
        # 연도순으로 정렬된 이미지 파일 목록
        imgs = sorted(glob.glob(os.path.join(folder, f"*_{t}_{method}.png")))

        if not imgs:
            print(f"⚠️ {t} - {method} : 이미지 없음, 건너뜀.")
            continue

        # 이미지 열기 및 PDF 저장
        image_list = [Image.open(f).convert("RGB") for f in imgs]
        pdf_path = os.path.join(folder, f"{t}_{method}_all.pdf")

        # 첫 장 + 나머지 묶기
        image_list[0].save(pdf_path, save_all=True, append_images=image_list[1:])
        print(f"✅ {t}_{method}_all.pdf 생성 완료 ({len(image_list)} pages)")


✅ avg_quadrant_all.pdf 생성 완료 (10 pages)
✅ avg_quantile70_all.pdf 생성 완료 (10 pages)
✅ avg_kmeans_all.pdf 생성 완료 (10 pages)
✅ max_quadrant_all.pdf 생성 완료 (10 pages)
✅ max_quantile70_all.pdf 생성 완료 (10 pages)
✅ max_kmeans_all.pdf 생성 완료 (10 pages)
✅ min_quadrant_all.pdf 생성 완료 (10 pages)
✅ min_quantile70_all.pdf 생성 완료 (10 pages)
✅ min_kmeans_all.pdf 생성 완료 (10 pages)


👉 2011–2020년 이미지들을 한 페이지(2×5)로 병합  
👉 avg / max / min 각각의 폴더에서  
👉 quadrant / quantile70 / kmeans 3가지 모두 처리  

In [21]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# ===== 기본 경로 =====
base_path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA"

# ===== 폴더 및 방법 =====
types = ["avg", "max", "min"]
methods = ["quadrant", "quantile70", "kmeans"]
years = range(2011, 2021)

for t in types:
    folder = os.path.join(base_path, t)
    if not os.path.exists(folder):
        print(f"⚠️ {folder} 폴더 없음 → 건너뜀.")
        continue

    for method in methods:
        imgs = sorted(glob.glob(os.path.join(folder, f"*_{t}_{method}.png")))
        if not imgs:
            print(f"⚠️ {t}-{method}: 이미지 없음 → 건너뜀.")
            continue

        # ===== Figure 설정 =====
        fig, axes = plt.subplots(2, 5, figsize=(20, 8))
        axes = axes.flatten()

        for i, (ax, year) in enumerate(zip(axes, years)):
            if i < len(imgs):
                img = mpimg.imread(imgs[i])
                ax.imshow(img)
                ax.set_title(f"{year}", fontsize=14)
                ax.axis('off')
            else:
                ax.axis('off')

        # ===== 제목 & 저장 =====
        title = method.replace("quantile70", "70th Percentile").title()
        plt.suptitle(f"{t.upper()} - {title} (2011–2020)", fontsize=18)
        plt.tight_layout(rect=[0, 0, 1, 0.95])

        save_path = os.path.join(folder, f"{t}_grid_{method}.png")
        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"✅ {save_path} 생성 완료 ({len(imgs)}개 연도)")


✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\avg\avg_grid_quadrant.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\avg\avg_grid_quantile70.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\avg\avg_grid_kmeans.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\max\max_grid_quadrant.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\max\max_grid_quantile70.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\max\max_grid_kmeans.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\min\min_grid_quadrant.png 생성 완료 (10개 연도)
✅ C:\Users\sl177\OneDrive - University of Illinois - Urbana\Code\Morris\SimlabSAUA\min\min_grid_quantile70.png 생성 완료 (10개 연도)
✅ C:\U